## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2


# 반도체5개 Δ이상치 및 ICR 분해 EDA

## 목적

1. **11번**: 2017년 반도체 슈퍼사이클(평시 트랙 사전추세검정에서 발견) 같은 패턴이, 실제로 쓰는 두 분석기간(긴축기 2018~2023, 평시 2015~2019)에서 **전년대비 변화량(Δ) 기준**으로도 나타나는지 확인한다. 지금까지는 절대값 순위(담배에 가려짐)만 봤고 Δ기준·분석기간 한정 확인은 안 한 상태였다.
2. **12번**: ICR이 왜 움직였는지 — `ICR = 영업이익 / 이자비용`이므로, 변화가 **영업이익 요인**인지 **이자비용 요인**인지 분해한다.

### 반도체 관련 5개 업종
`I3102(반도체) I3103(디스플레이) I3104(컴퓨터) I3105(통신기기) I3108(전지)`

**주의(검토8원칙 4번)**: 이 5개는 ISTANS 공식분류(I31 고위기술산업군 9개)의 부분집합이며, 과거 2017년 ICR 급등 패턴을 보고 사후에 고른 것이다. 여기서도 그 그룹을 그대로 재사용해 확인하지만, 최종적으로 반도체 통제변수를 만들 때는 이 그룹 정의 자체의 임의성을 다시 점검해야 한다(15번 항목 참고).


## 1. 데이터 로드

In [2]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Pretendard", "Noto Sans CJK KR"]:
    try:
        mpl.font_manager.findfont(f, fallback_to_default=False)
        mpl.rcParams["font.family"] = f
        break
    except Exception:
        pass
mpl.rcParams["axes.unicode_minus"] = False

# "2. EDA/" 폴더에서 실행하는 것을 기준으로 상대경로를 잡음
BASE_DATA = INTERIM
BASE_MERGE = PROCESSED

p_stab_full = BASE_DATA / "ISTANS 안정성" / "안정성_전체통합_1994_2025.csv"
p_prof_full = BASE_DATA / "ISTANS 수익성" / "수익성_전체통합_1994_2025.csv"
p_tight = BASE_MERGE / "재무_원자재_병합패널.csv"
p_calm  = BASE_MERGE / "평시_재무_원자재_병합패널.csv"

for label, p in [("안정성 전체", p_stab_full), ("수익성 전체", p_prof_full),
                  ("긴축기 병합", p_tight), ("평시 병합", p_calm)]:
    if not p.exists():
        raise FileNotFoundError(f"{label} 파일을 못 찾음: {p.resolve()}")

stab_full = pd.read_csv(p_stab_full, encoding="utf-8-sig")
prof_full = pd.read_csv(p_prof_full, encoding="utf-8-sig")
tight = pd.read_csv(p_tight, encoding="utf-8-sig")
calm  = pd.read_csv(p_calm,  encoding="utf-8-sig")

SEMI5 = ["I3102", "I3103", "I3104", "I3105", "I3108"]
SEMI5_NAMES = {"I3102":"반도체", "I3103":"디스플레이", "I3104":"컴퓨터", "I3105":"통신기기", "I3108":"전지"}

print("안정성 전체:", stab_full.shape)
print("수익성 전체:", prof_full.shape)
print("긴축기 병합:", tight.shape, "|", tight["연도"].min(), "~", tight["연도"].max())
print("평시 병합  :", calm.shape,  "|", calm["연도"].min(), "~", calm["연도"].max())


안정성 전체: (1280, 21)
수익성 전체: (1280, 8)
긴축기 병합: (240, 16) | 2018 ~ 2023
평시 병합  : (200, 16) | 2015 ~ 2019


## 2. Δ ICR (전년대비 변화량) 계산

`수익성_전체통합_1994_2025.csv`(전체 기간 보유)에서 필요한 연도(분석기간 + 그 직전 1개년)를 뽑아 Δ를 계산한다. 분석기간 첫 해(긴축기 2018, 평시 2015)도 Δ를 구하려면 그 직전 해(2017, 2014) 값이 필요하기 때문이다.

In [3]:
def build_delta_icr(years, label):
    need_years = list(range(min(years) - 1, max(years) + 1))
    sub = prof_full[prof_full["연도"].isin(need_years)][
        ["업종코드", "업종", "연도", "이자보상비율"]
    ].copy().sort_values(["업종코드", "연도"])

    sub["ICR_변화"] = sub.groupby("업종코드")["이자보상비율"].diff()
    sub = sub[sub["연도"].isin(years)].copy()
    sub["반도체5"] = sub["업종코드"].isin(SEMI5)
    sub["트랙"] = label
    return sub

delta_tight = build_delta_icr(sorted(tight["연도"].unique()), "긴축기")
delta_calm  = build_delta_icr(sorted(calm["연도"].unique()),  "평시")

print("긴축기 Δ건수:", len(delta_tight), "| 결측(첫해):", delta_tight["ICR_변화"].isna().sum())
print("평시   Δ건수:", len(delta_calm),  "| 결측(첫해):", delta_calm["ICR_변화"].isna().sum())


긴축기 Δ건수: 240 | 결측(첫해): 0
평시   Δ건수: 200 | 결측(첫해): 0


## 3. 연도별 상위·하위 Δ ICR — 반도체5개가 자주 등장하는지 확인

In [4]:
def year_rank_table(delta_df, n=8):
    out = []
    for y, g in delta_df.dropna(subset=["ICR_변화"]).groupby("연도"):
        top = g.nlargest(n, "ICR_변화")[["업종코드","업종","ICR_변화","반도체5"]]
        top["구분"] = "상위"
        bot = g.nsmallest(n, "ICR_변화")[["업종코드","업종","ICR_변화","반도체5"]]
        bot["구분"] = "하위"
        combo = pd.concat([top, bot])
        combo["연도"] = y
        out.append(combo)
    return pd.concat(out, ignore_index=True)

print("=== 긴축기 연도별 Δ ICR 상하위 ===")
rank_tight = year_rank_table(delta_tight)
display(rank_tight)

print("\n반도체5 비중(상위8/하위8 중): ")
display(rank_tight.groupby(["연도","구분"])["반도체5"].mean().unstack())


=== 긴축기 연도별 Δ ICR 상하위 ===


,업종코드,업종,ICR_변화,반도체5,구분,연도
0,I3102,반도체,"2,969.7000",True,상위,2018
1,I3108,전지,744.1000,True,상위,2018
2,I3105,통신기기,694.8000,True,상위,2018
3,I3109,항공,401.5000,False,상위,2018
4,I3209,기타 수송장비,278.1000,False,상위,2018
...,...,...,...,...,...,...
91,I3301,석유정제,"-1,030.2000",False,하위,2023
92,I3203,기타 전자부품,-922.5000,False,하위,2023
93,I3107,정밀기기,-640.7000,False,하위,2023
94,I3309,비철금속,-596.2000,False,하위,2023



반도체5 비중(상위8/하위8 중): 


구분,상위,하위
연도,,
2018,0.3750,0.1250
2019,0.0000,0.6250
2020,0.5000,0.1250
2021,0.3750,0.1250
2022,0.0000,0.3750
2023,0.1250,0.2500


In [5]:
print("=== 평시 연도별 Δ ICR 상하위 ===")
rank_calm = year_rank_table(delta_calm)
display(rank_calm)

print("\n반도체5 비중(상위8/하위8 중): ")
display(rank_calm.groupby(["연도","구분"])["반도체5"].mean().unstack())


=== 평시 연도별 Δ ICR 상하위 ===


,업종코드,업종,ICR_변화,반도체5,구분,연도
0,I3402,담배,"12,769.6000",False,상위,2015
1,I3301,석유정제,911.9000,False,상위,2015
2,I3103,디스플레이,885.4000,True,상위,2015
3,I3201,석유화학,603.3000,False,상위,2015
4,I3102,반도체,476.8000,True,상위,2015
...,...,...,...,...,...,...
75,I3103,디스플레이,"-1,177.2000",True,하위,2019
76,I3108,전지,-789.6000,True,하위,2019
77,I3201,석유화학,-750.8000,False,하위,2019
78,I3301,석유정제,-644.7000,False,하위,2019



반도체5 비중(상위8/하위8 중): 


구분,상위,하위
연도,,
2015,0.2500,0.2500
2016,0.0000,0.5000
2017,0.6250,0.0000
2018,0.3750,0.1250
2019,0.0000,0.6250


### 해석

반도체5 비중(상위8/하위8 중, 무작위라면 5/40=12.5%가 나와야 정상):

| 연도(긴축기) | 상위 | 하위 | | 연도(평시) | 상위 | 하위 |
|---|---|---|---|---|---|---|
| 2018 | 37.5% | 12.5% | | 2015 | 25.0% | 25.0% |
| 2019 | 0% | 62.5% | | 2016 | 0% | 50.0% |
| 2020 | 50.0% | 12.5% | | 2017 | 62.5% | 0% |
| 2021 | 37.5% | 12.5% | | 2018 | 37.5% | 12.5% |
| 2022 | 0% | 37.5% | | 2019 | 0% | 62.5% |
| 2023 | 12.5% | 25.0% | | | | |

평시 2017년 상위 62.5%는 이미 알던 반도체 슈퍼사이클이 Δ기준으로도 그대로 잡힌 거고(무작위 기대치의 5배니까 우연은 아님), 긴축기 2019년은 반대로 하위에 62.5%가 몰려있다 — 이건 새로 확인된 건데, 2019년에 반도체5개가 다같이 ICR 급락 쪽으로 갔다는 뜻이다(그 해 메모리반도체 가격이 많이 빠졌던 시기와 맞음).

더 눈에 띄는 건 2020~21(상위 편중) → 2022(하위 편중)로 이어지는 흐름이다. 2017년 한 번 튄 게 아니라 긴축기 내내 계속 극단값 쪽에 몰리는 걸 보면, 반도체 업황 자체가 이 그룹 ICR을 구조적으로 계속 흔든다고 보는 게 맞을 듯 — 반도체 통제변수가 필요하다는 근거가 하나 더 생긴 셈.

## 4. 반도체5개 포함/제외 평균 Δ 비교 (leave-group-out)

`02_사전추세_검정.ipynb` 부록에서 쓴 것과 같은 방법 — 그룹을 통째로 빼고 평균이 얼마나 바뀌는지 본다.

In [6]:
def group_effect(delta_df, label):
    rows = []
    for y, g in delta_df.dropna(subset=["ICR_변화"]).groupby("연도"):
        mean_all = g["ICR_변화"].mean()
        mean_excl = g.loc[~g["반도체5"], "ICR_변화"].mean()
        mean_semi5 = g.loc[g["반도체5"], "ICR_변화"].mean()
        rows.append({
            "연도": y, "전체평균": mean_all, "반도체5제외평균": mean_excl,
            "반도체5평균": mean_semi5, "차이(전체-제외)": mean_all - mean_excl
        })
    out = pd.DataFrame(rows)
    print(f"=== {label} ===")
    display(out.round(2))
    return out

ge_tight = group_effect(delta_tight, "긴축기")


=== 긴축기 ===


,연도,전체평균,반도체5제외평균,반도체5평균,차이(전체-제외)
0,2018,-236.2800,-283.9400,97.3400,47.6600
1,2019,-378.5600,-24.4700,"-2,857.2000",-354.0900
2,2020,249.4600,143.1400,993.6800,106.3200
3,2021,458.1200,362.2500,"1,129.2800",95.8800
4,2022,-298.5200,-162.8800,"-1,247.9800",-135.6400
5,2023,-573.7600,-440.3700,"-1,507.5200",-133.3900


In [7]:
ge_calm = group_effect(delta_calm, "평시")


=== 평시 ===


,연도,전체평균,반도체5제외평균,반도체5평균,차이(전체-제외)
0,2015,411.9500,466.2900,31.5400,-54.3400
1,2016,90.3200,183.3900,-561.1600,-93.0700
2,2017,-208.1900,-617.7200,"2,658.5000",409.5300
3,2018,-236.2800,-283.9400,97.3400,47.6600
4,2019,-378.5600,-24.4700,"-2,857.2000",-354.0900


### 해석

차이(전체평균-반도체5제외평균)가 클수록 그 해엔 반도체5개가 전체 평균을 세게 끌고 간 거다:

| 연도(긴축기) | 차이 | | 연도(평시) | 차이 |
|---|---|---|---|---|
| 2018 | +47.66 | | 2015 | -54.34 |
| 2019 | -354.09 | | 2016 | -93.07 |
| 2020 | +106.32 | | 2017 | +409.53 |
| 2021 | +95.88 | | 2018 | +47.66 |
| 2022 | -135.64 | | 2019 | -354.09 |
| 2023 | -133.39 | | | |

평시 2017년 +409.53이 전체 중 압도적으로 제일 크다 — 반도체 슈퍼사이클 효과가 여기서도 그대로 재현됨. 2018·2019 숫자가 두 트랙에서 완전히 똑같이 나온 건(당연하지만) 두 트랙이 실제로 겹치는 해라서 그런 거고, 계산이 서로 맞다는 교차검증도 되는 셈.

제일 중요한 건 긴축기 2022(-135.64)·2023(-133.39)이다. 우리가 걱정했던 "반도체 다운턴이 2022~23년 계수를 오염시킬 수 있다"는 얘기가 실제 숫자로 나온 거라, 반도체 통제를 안 넣으면 노출도 효과 추정치에 이 영향이 섞일 위험이 진짜 있다고 봐야 한다.

## 5. ICR 분해 — 영업이익 요인 vs 이자비용 요인

`ICR = 영업이익 / 이자비용`이므로, `영업이익`(수익성 전체통합)과 `이자비용`(안정성 전체통합)을 업종×연도로 조인해서 각각의 전년대비 변화율을 따로 본다.

In [8]:
def build_decomp(years, label):
    need_years = list(range(min(years) - 1, max(years) + 1))

    op = prof_full[prof_full["연도"].isin(need_years)][
        ["업종코드", "업종", "연도", "영업이익", "이자보상비율"]
    ].copy()
    ie = stab_full[stab_full["연도"].isin(need_years)][
        ["업종코드", "연도", "이자비용"]
    ].copy()

    d = op.merge(ie, on=["업종코드", "연도"], how="left", validate="one_to_one")
    d = d.sort_values(["업종코드", "연도"])

    d["영업이익_증가율"] = d.groupby("업종코드")["영업이익"].pct_change() * 100
    d["이자비용_증가율"] = d.groupby("업종코드")["이자비용"].pct_change() * 100
    d["ICR_변화"] = d.groupby("업종코드")["이자보상비율"].diff()

    d = d[d["연도"].isin(years)].copy()
    d["반도체5"] = d["업종코드"].isin(SEMI5)
    d["트랙"] = label
    return d

decomp_tight = build_decomp(sorted(tight["연도"].unique()), "긴축기")
decomp_calm  = build_decomp(sorted(calm["연도"].unique()),  "평시")

print("=== 긴축기: 반도체5개 업종의 ICR 변화 vs 영업이익/이자비용 증가율 ===")
display(
    decomp_tight[decomp_tight["반도체5"]]
    [["업종", "연도", "ICR_변화", "영업이익_증가율", "이자비용_증가율"]]
    .sort_values(["업종", "연도"])
)


=== 긴축기: 반도체5개 업종의 ICR 변화 vs 영업이익/이자비용 증가율 ===


,업종,연도,ICR_변화,영업이익_증가율,이자비용_증가율
15,디스플레이,2018,"-3,695.3000",-89.0512,-5.0000
16,디스플레이,2019,"-1,177.2000",-353.9305,74.5614
17,디스플레이,2020,821.8000,-123.4701,33.1658
18,디스플레이,2021,"1,357.2000","1,103.9877",0.0000
19,디스플레이,2022,-994.3000,-65.9873,3.7736
20,디스플레이,2023,-503.6000,-106.9663,90.1818
8,반도체,2018,"2,969.7000",53.9310,-5.0847
9,반도체,2019,"-6,916.0000",-85.0982,41.4286
10,반도체,2020,781.6000,85.9181,-5.0505
11,반도체,2021,"1,861.1000",147.0971,14.0957


In [9]:
print("=== 평시: 반도체5개 업종의 ICR 변화 vs 영업이익/이자비용 증가율 ===")
display(
    decomp_calm[decomp_calm["반도체5"]]
    [["업종", "연도", "ICR_변화", "영업이익_증가율", "이자비용_증가율"]]
    .sort_values(["업종", "연도"])
)


=== 평시: 반도체5개 업종의 ICR 변화 vs 영업이익/이자비용 증가율 ===


,업종,연도,ICR_변화,영업이익_증가율,이자비용_증가율
13,디스플레이,2015,885.4000,89.5156,-11.7347
14,디스플레이,2016,-69.1000,-13.7605,-10.4046
15,디스플레이,2017,"2,589.5000",102.8421,-22.5806
16,디스플레이,2018,"-3,695.3000",-89.0512,-5.0000
17,디스플레이,2019,"-1,177.2000",-353.9305,74.5614
7,반도체,2015,476.8000,6.8161,-21.4286
8,반도체,2016,-633.4000,-39.0385,-5.9561
9,반도체,2017,"3,598.1000",303.0685,-1.6667
10,반도체,2018,"2,969.7000",53.9310,-5.0847
11,반도체,2019,"-6,916.0000",-85.0982,41.4286


### 해석

반도체5개 대부분 연도는 영업이익 증가율이 이자비용 증가율보다 압도적으로 크게 움직인다 — 반도체 2017(영업이익 +303.1%, 이자비용 -1.7%), 통신기기 2017(+162.3%, +4.1%), 디스플레이 2021(+1104.0%, 0.0%) 같은 식. ICR 변화는 대체로 영업이익(분자) 요인이라던 기존 얘기랑 일치한다.

근데 예외가 있다 — 2019년, 그리고 특히 2022~2023년엔 이자비용도 꽤 커진다:
- 반도체 2019: 영업이익 -85.1%, 이자비용 +41.4%
- 통신기기 2023: 영업이익 -140.8%, 이자비용 +61.5%
- 반도체 2023: 영업이익 -131.3%, **이자비용 +139.8%** — 이자비용 증가율이 영업이익 감소율만큼 크다

그러니까 "반도체 ICR변화는 다 영업이익 때문"이라고 단순하게 말하면 안 되고, 긴축기 후반부엔 금리인상 효과가 실제로 이자비용에 반영되기 시작한 걸로 보인다. 이것도 반도체 통제가 왜 필요한지 보여주는 증거.

## 6. 정리

- 11번: 반도체5개는 2017년(평시)뿐 아니라 긴축기 전체에 걸쳐(2019 급락, 2020~21 급등, 2022 하락) 계속 극단값에 몰린다 — 반도체 경기순환이 구조적으로 작용하는 걸로 보임.
- 12번: ICR 변화는 대부분 영업이익 때문인데, 2019·2022~23년엔 이자비용도 상당히 커짐(특히 반도체 2023).
- 둘 다 반도체 통제변수가 필요하다는 근거 → 15번으로 이어짐.
- 다음: 13번(선형성), 14번(wild bootstrap).
